In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os
from PIL import Image
import logging
import time

# ===== CONFIG =====
TEST_DIR = "C:/DUT/Ki 2/Edge AI/dataset/test/"
IMG_SIZE = 96

MODEL_TYPE = "tflite"  # "h5" hoặc "tflite"
MODEL_PATH = "model.tflite"  # hoặc "model.h5"

# ===== LOGGER =====
logging.basicConfig(
    filename="predict.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("===== START PREDICTION =====")

# ===== PREPROCESS =====
def preprocess(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img = np.array(img) / 255.0
    return img.astype(np.float32)

# ===== LOAD MODEL =====
if MODEL_TYPE == "h5":
    logging.info("Loading H5 model...")
    model = tf.keras.models.load_model(MODEL_PATH)

elif MODEL_TYPE == "tflite":
    logging.info("Loading TFLite model...")
    interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

else:
    raise ValueError("MODEL_TYPE must be 'h5' or 'tflite'")

# ===== PREDICT =====
ids = []
preds = []

start_time = time.time()

files = os.listdir(TEST_DIR)
logging.info(f"Total test images: {len(files)}")

for i, file in enumerate(files):
    path = os.path.join(TEST_DIR, file)

    try:
        img = preprocess(path)
        img = np.expand_dims(img, axis=0)

        if MODEL_TYPE == "h5":
            output = model.predict(img, verbose=0)
            pred = np.argmax(output)

        else:  # TFLite
            interpreter.set_tensor(input_details[0]['index'], img)
            interpreter.invoke()
            output = interpreter.get_tensor(output_details[0]['index'])
            pred = np.argmax(output)

        ids.append(file)
        preds.append(pred)

        if i % 50 == 0:
            logging.info(f"Processed {i}/{len(files)} images")

    except Exception as e:
        logging.error(f"Error with file {file}: {str(e)}")

end_time = time.time()

logging.info(f"Prediction finished in {end_time - start_time:.2f} seconds")

# ===== SAVE CSV =====
df = pd.DataFrame({
    "id": ids,
    "label": preds
})

df.to_csv("submission.csv", index=False)

logging.info("Saved submission.csv")
logging.info("===== END =====")

print("Done! Check submission.csv and predict.log")